# Data and baselines
Each **M** cell is one row in the tracker.
Outputs go to the shared Drive folder `lost-at-the-switch/`.

**One-time setup before cell S:**
1. Accept the GitHub invite to `Narpear/lost-at-the-switch`, then make a **classic** token (github.com -> Settings -> Developer settings -> Tokens (classic), scope `repo`).
2. Make a Hugging Face **Read** token, and click "Agree and access" on https://huggingface.co/datasets/openlanguagedata/flores_plus
3. In Colab's key icon add `GITHUB_TOKEN` and `HF_TOKEN`, notebook access ON.
4. In Google Drive: Shared with me -> `lost-at-the-switch` -> Organize -> Add shortcut -> My Drive.

### S. Setup (run every time you open a new runtime)
Runtime -> Change runtime type -> **A100**, else **L4** (T4 is too small).
Secrets (key icon, notebook access ON): `GITHUB_TOKEN`, `HF_TOKEN`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, os
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type"
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/lost-at-the-switch'
if not os.path.isdir(DRIVE_DIR):
    raise SystemExit("Shared folder not found. In Google Drive: 'Shared with me' -> right-click "
                     "'lost-at-the-switch' -> Organize -> Add shortcut -> My Drive. Then rerun this cell.")
for sub in ['results', 'data/raw', 'data/processed']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_HOME'] = '/content/hf_cache'
REPO = '/content/lost-at-the-switch'
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} pull --ff-only
else:
    !git clone -q https://{GITHUB_TOKEN}@github.com/Narpear/lost-at-the-switch.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q -e ".[dev]"
# data/ and results/ live in the shared Drive folder, so both of us see everything
!rm -rf results data/raw data/processed
!ln -s {DRIVE_DIR}/results results && ln -s {DRIVE_DIR}/data/raw data/raw && ln -s {DRIVE_DIR}/data/processed data/processed
!ls -la results data/raw data/processed | head -20

## M1. Get LinCE (Hindi-English, Spanish-English, Nepali-English)
Tries the official server first. If it times out (it did on Monday): download LinCE yourself
(https://ritual.uh.edu/lince, or a classmate's copy), upload the zip/folder to the shared Drive folder,
and run the second cell with its path.

In [ ]:
!python scripts/fetch_lince.py --timeout 120

In [ ]:
# Only if the cell above failed: point --from at your downloaded zip or folder
# !python scripts/fetch_lince.py --from /content/drive/MyDrive/lost-at-the-switch/lince_download.zip

In [ ]:
!python scripts/data_stats.py data/raw/lince/lid_hineng/dev.conll data/raw/lince/lid_spaeng/dev.conll data/raw/lince/lid_nepeng/dev.conll | tee results/data_stats_lince_dev.txt

## M2. FLORES-200 monolingual controls + English->X translation prompts
Needs the FLORES access click (setup step 2). Untested from our side: if it errors, paste the error to Claude.

In [ ]:
!python scripts/fetch_flores.py --split devtest

## M3. Switching-penalty control: teacher-forced acceptance on monolingual FLORES
Same measurement as P3 but on text with no switches. 500 sentences per language.

In [ ]:
for lang in ['eng_Latn', 'spa_Latn', 'hin_Deva', 'npi_Deva']:
    print('=' * 20, lang)
    !python scripts/run_positional.py --pair qwen --data data/raw/flores/{lang}.conll --out results/positional/qwen_flores_{lang}.jsonl --limit 500 --skip-existing 2>&1 | grep -v -E "falling back|Loading|it/s" | tail -12

## M4. Make code-switched prompts (same files as P4; skipped if they exist)

In [ ]:
# Code-switched prompts from LinCE dev, in two styles with the same ids:
#   *_dev_prompts.jsonl            raw continuation (the model drifts off code-switching)
#   *_dev_prompts_instruct.jsonl   "continue in the same mixed style", used with --chat-template
for cfg in ['lid_hineng', 'lid_spaeng', 'lid_nepeng']:
    for style, suffix in [('continuation', ''), ('instruct', '_instruct')]:
        out = f'data/processed/{cfg}_dev_prompts{suffix}.jsonl'
        if not os.path.exists(out):
            !python scripts/make_prompts.py data/raw/lince/{cfg}/dev.conll {out} --style {style}
        else:
            print(out, 'exists')

## M5. Bigram drafter tables (B2), built from LinCE train with the Qwen tokenizer

In [ ]:
for cfg in ['lid_hineng', 'lid_spaeng', 'lid_nepeng']:
    pair = cfg.replace('lid_', '')
    !python scripts/build_ngram.py --pair qwen --n 2 --inputs data/raw/lince/{cfg}/train.conll --out results/ngram/qwen_{pair}_n2.pkl

## M6. B2 on code-switched text: bigram drafter, instruct prompts + chat template (same prompts as P5)
5-prompt check first, then the full run. Resumes automatically if Colab disconnects.

In [ ]:
# Quick check on 5 prompts per pair: do generations stay code-switched?
# Compare with real LinCE text (--field reference) and, if present, the raw-continuation runs.
for cfg in ['lid_hineng', 'lid_spaeng', 'lid_nepeng']:
    pair = cfg.replace('lid_', '')
    print('=' * 20, pair)
    !python scripts/run_specdec.py --pair qwen --ngram results/ngram/qwen_{pair}_n2.pkl --prompts data/processed/{cfg}_dev_prompts_instruct.jsonl --out results/checks/b2_{pair}_instruct.jsonl --limit 5 --gammas 3 --chat-template --skip-existing 2>&1 | grep -v -E "falling back|Loading|it/s" | tail -3
    !python scripts/cs_check.py --train data/raw/lince/{cfg}/train.conll --field reference data/processed/{cfg}_dev_prompts.jsonl
    raw = f'results/specdec/qwen_b2_{pair}.jsonl'          # raw-continuation run, if one exists
    raw = raw if os.path.exists(raw) else ''
    !python scripts/cs_check.py --train data/raw/lince/{cfg}/train.conll --show 3 results/checks/b2_{pair}_instruct.jsonl {raw}

In [ ]:
for cfg in ['lid_hineng', 'lid_spaeng', 'lid_nepeng']:
    pair = cfg.replace('lid_', '')
    print('=' * 20, pair)
    !python scripts/run_specdec.py --pair qwen --ngram results/ngram/qwen_{pair}_n2.pkl --prompts data/processed/{cfg}_dev_prompts_instruct.jsonl --out results/specdec/qwen_b2_{pair}_instruct.jsonl --limit 50 --gammas 2 3 4 --chat-template --skip-existing 2>&1 | grep -v -E "falling back|Loading|it/s" | tail -30
    !python scripts/cs_check.py --train data/raw/lince/{cfg}/train.conll results/specdec/qwen_b2_{pair}_instruct.jsonl

## M7. Tables + figure for Friday

In [ ]:
!python scripts/make_report.py
from IPython.display import Image, display
if os.path.exists('results/report/positional.png'): display(Image('results/report/positional.png'))